# 05 · Monitoring: Infrastructure, Data Quality and Model Quality
**AAI-540 Group 3 — CloudTrail Suspicious-Activity Classifier**

| | |
|---|---|
| **Goal** | Monitor the deployed system on three layers and show that the monitors fire when something goes wrong. |
| **Input** | Endpoint `cloudtrail-risk-endpoint` + data capture (notebook 04), training baseline (notebook 03) |
| **Output** | CloudWatch dashboard + alarms · SageMaker Model Monitor baseline, schedule and violation report · offline drift / performance report (charts below) |
| **Last step** | **Cleanup** — deletes the endpoint and monitoring schedule so they stop billing |

| layer | what we watch | tool |
|---|---|---|
| **Infrastructure** | invocations, 4xx / 5xx errors, model latency, CPU / memory | CloudWatch dashboard + alarms |
| **Data** | schema, missing values, distribution drift of each input feature vs. training | SageMaker Model Monitor (data quality) + PSI drift report |
| **Model** | score drift, alert volume vs. analyst budget, recall once (delayed) labels arrive | offline monitoring report (Section 4) |

## 1. Setup

In [ ]:
%%capture pip_log
# Install the project's pinned libraries (run once per kernel, BEFORE any other import).
# The pin keeps the SageMaker Python SDK on v2, which all notebooks in this repo use.
# Output is captured to keep the notebook clean; run print(pip_log.stdout) to see it.
# (Warnings about sagemaker-core / sagemaker-mlops conflicts refer to Studio's pre-installed
#  SDK v3 packages, which this project does not use.)
%pip install -q -r ../requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))   # shared project package: src/cloudtrail_mlops
DATA_DIR = Path.home() / "cloudtrail-data"           # local working copy, outside the git repo
DATA_DIR.mkdir(exist_ok=True)
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"         # we intentionally pin SDK v2 (see requirements.txt)

import json
import time
from datetime import datetime, timedelta, timezone

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sagemaker.model_monitor import CronExpressionGenerator, DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sklearn.metrics import average_precision_score

from cloudtrail_mlops import aws, config, dataset, evaluation, model
from cloudtrail_mlops.plots import BLUE, MUTED, ORANGE, SERIES, setup_style

setup_style()
ctx = aws.get_context()
state = aws.load_state(ctx)
sm_client = ctx.session.sagemaker_client
cloudwatch = boto3.client("cloudwatch", region_name=ctx.region)
runtime = boto3.client("sagemaker-runtime", region_name=ctx.region)

ENDPOINT_NAME = state["endpoint_name"]
thresholds = state["thresholds"]
print(ENDPOINT_NAME, "->", sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"])

encoder = dataset.FeatureEncoder.load(DATA_DIR / "model-input" / "encoder.json")
mart = dataset.load_mart(DATA_DIR / "features" / "feature_mart.parquet")
splits = dataset.chronological_split(mart)


def invoke(windows: pd.DataFrame) -> np.ndarray:
    payload = encoder.transform(windows).to_csv(header=False, index=False, float_format="%.6g")
    body = runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType="text/csv", Body=payload)["Body"].read().decode()
    return np.array([float(v) for v in body.replace("\n", ",").split(",") if v.strip()])

## 2. Infrastructure monitoring — CloudWatch dashboard and alarms
SageMaker publishes endpoint metrics to CloudWatch automatically. We build one dashboard and three alarms:

| alarm | condition | why |
|---|---|---|
| `…-5xx-errors` | any 5xx error in 5 minutes | the model container is failing |
| `…-latency-p95` | p95 model latency > 200 ms for 15 minutes | alerts would arrive late |
| `…-cpu-high` | average CPU > 80 % for 15 minutes | instance too small → scale out |

In [ ]:
DASHBOARD = "cloudtrail-risk-endpoint"
dimensions = ["EndpointName", ENDPOINT_NAME, "VariantName", "AllTraffic"]


def widget(x, y, title, metrics):
    return {"type": "metric", "x": x, "y": y, "width": 8, "height": 6,
            "properties": {"title": title, "region": ctx.region, "view": "timeSeries", "period": 60, "metrics": metrics}}


dashboard = {"widgets": [
    widget(0, 0, "Invocations per minute", [["AWS/SageMaker", "Invocations", *dimensions, {"stat": "Sum"}]]),
    widget(8, 0, "Errors (4XX / 5XX)", [["AWS/SageMaker", "Invocation4XXErrors", *dimensions, {"stat": "Sum"}],
                                       ["AWS/SageMaker", "Invocation5XXErrors", *dimensions, {"stat": "Sum"}]]),
    widget(16, 0, "Model latency (microseconds)", [["AWS/SageMaker", "ModelLatency", *dimensions, {"stat": "p50"}],
                                                   ["AWS/SageMaker", "ModelLatency", *dimensions, {"stat": "p95"}]]),
    widget(0, 6, "CPU utilisation (%)", [["/aws/sagemaker/Endpoints", "CPUUtilization", *dimensions, {"stat": "Average"}]]),
    widget(8, 6, "Memory utilisation (%)", [["/aws/sagemaker/Endpoints", "MemoryUtilization", *dimensions, {"stat": "Average"}]]),
    widget(16, 6, "Disk utilisation (%)", [["/aws/sagemaker/Endpoints", "DiskUtilization", *dimensions, {"stat": "Average"}]]),
]}
cloudwatch.put_dashboard(DashboardName=DASHBOARD, DashboardBody=json.dumps(dashboard))

alarm_dims = [{"Name": "EndpointName", "Value": ENDPOINT_NAME}, {"Name": "VariantName", "Value": "AllTraffic"}]
alarms = [
    dict(AlarmName=f"{ENDPOINT_NAME}-5xx-errors", Namespace="AWS/SageMaker", MetricName="Invocation5XXErrors",
         Statistic="Sum", Period=300, EvaluationPeriods=1, Threshold=0, ComparisonOperator="GreaterThanThreshold"),
    dict(AlarmName=f"{ENDPOINT_NAME}-latency-p95", Namespace="AWS/SageMaker", MetricName="ModelLatency",
         ExtendedStatistic="p95", Period=300, EvaluationPeriods=3, Threshold=200_000, ComparisonOperator="GreaterThanThreshold"),
    dict(AlarmName=f"{ENDPOINT_NAME}-cpu-high", Namespace="/aws/sagemaker/Endpoints", MetricName="CPUUtilization",
         Statistic="Average", Period=300, EvaluationPeriods=3, Threshold=80, ComparisonOperator="GreaterThanThreshold"),
]
for alarm in alarms:
    cloudwatch.put_metric_alarm(Dimensions=alarm_dims, TreatMissingData="notBreaching", **alarm)
aws.save_state(ctx, dashboard=DASHBOARD, alarms=[a["AlarmName"] for a in alarms])

print("Dashboard:", f"https://{ctx.region}.console.aws.amazon.com/cloudwatch/home?region={ctx.region}#dashboards:name={DASHBOARD}")
pd.DataFrame(cloudwatch.describe_alarms(AlarmNames=[a["AlarmName"] for a in alarms])["MetricAlarms"])[
    ["AlarmName", "MetricName", "Threshold", "ComparisonOperator", "StateValue"]]

The same metrics, pulled into the notebook (traffic generated in notebook 04 and in Section 3 below):

In [ ]:
def endpoint_metric(namespace, name, stat, hours=3):
    end = datetime.now(timezone.utc)
    kwargs = {"ExtendedStatistics": [stat]} if stat.startswith("p") else {"Statistics": [stat]}
    points = cloudwatch.get_metric_statistics(Namespace=namespace, MetricName=name, Dimensions=alarm_dims,
                                              StartTime=end - timedelta(hours=hours), EndTime=end, Period=60, **kwargs)["Datapoints"]
    frame = pd.DataFrame(points)
    if frame.empty:
        return pd.Series(dtype=float)
    value = frame["ExtendedStatistics"].map(lambda d: d[stat]) if stat.startswith("p") else frame[stat]
    return pd.Series(value.values, index=frame["Timestamp"]).sort_index()


panels = [("AWS/SageMaker", "Invocations", "Sum", "Invocations per minute"),
          ("AWS/SageMaker", "ModelLatency", "p95", "Model latency p95 (µs)"),
          ("/aws/sagemaker/Endpoints", "CPUUtilization", "Average", "CPU utilisation (%)")]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
for ax, (namespace, name, stat, title) in zip(axes, panels):
    series = endpoint_metric(namespace, name, stat)
    if series.empty:
        ax.text(0.5, 0.5, "no datapoints yet\n(CloudWatch lags 1-3 min)", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.plot(series.index, series.values, color=BLUE, marker="o", markersize=3)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 3. Data-quality monitoring — SageMaker Model Monitor
1. **Baseline:** a processing job profiles the training features (`model-input/baseline/train_with_header.csv`) and suggests **statistics** (mean, std, distribution sketch per feature) and **constraints** (data types, completeness, value ranges).
2. **Live traffic:** every request to the endpoint is captured to S3 (enabled in notebook 04).
3. **Monitoring job:** compares the captured requests with the baseline and writes a **constraint-violations report**; violations are also published as CloudWatch metrics.

The baseline job takes ~5 minutes.

In [ ]:
monitor = DefaultModelMonitor(role=ctx.role, instance_count=1, instance_type="ml.m5.large", volume_size_in_gb=20,
                              max_runtime_in_seconds=1800, sagemaker_session=ctx.session)
monitor.suggest_baseline(baseline_dataset=f"{state['model_input_uri']}/baseline/train_with_header.csv",
                         dataset_format=DatasetFormat.csv(header=True),
                         output_s3_uri=ctx.s3_uri("monitoring", "baseline"), wait=True, logs=False)

baseline_stats = pd.json_normalize(monitor.baseline_statistics().body_dict["features"])
constraints = pd.json_normalize(monitor.suggested_constraints().body_dict["features"])
display(baseline_stats[["name", "inferred_type", "numerical_statistics.mean", "numerical_statistics.std_dev",
                        "numerical_statistics.min", "numerical_statistics.max"]].head(10).round(3))
constraints.head(10)

### 3.1 Send normal and **drifted** traffic
To prove the monitor works we send two kinds of traffic:
* **normal** — real 2020 windows;
* **drifted** — the same windows with activity counts multiplied by 6 and an AWS region the model has never seen. This imitates an upstream change (e.g. a new automation tool or a log-format change) that should trigger violations.

In [ ]:
normal_traffic = splits["test"].sample(1500, random_state=11)
drifted_traffic = splits["test"].sample(500, random_state=12).copy()
for column in ("distinct_apis", "error_count", "auth_error_count", "write_count", "prior_writes_24h", "distinct_source_ips"):
    drifted_traffic[column] = drifted_traffic[column].fillna(0) * 6
drifted_traffic["last_region"] = "ap-unseen-1"

# one window per request: each captured request is then exactly one record for Model Monitor
normal_scores = np.concatenate([invoke(normal_traffic.iloc[[i]]) for i in range(len(normal_traffic))])
drifted_scores = np.concatenate([invoke(drifted_traffic.iloc[[i]]) for i in range(len(drifted_traffic))])
print(f"normal traffic : {len(normal_scores)} windows | alert rate {np.mean(normal_scores >= thresholds['medium']):.1%}")
print(f"drifted traffic: {len(drifted_scores)} windows | alert rate {np.mean(drifted_scores >= thresholds['medium']):.1%}")
print("Waiting 3 minutes for data capture files to land in S3 ...")
time.sleep(180)

### 3.2 Run the data-quality monitor
We create a **one-off monitoring schedule** (`NOW`) that analyses the last hour of captured traffic immediately, instead of waiting for the next hourly run. (For production you would use `CronExpressionGenerator.hourly()` — the commented line below.) The execution takes ~10 minutes.

In [ ]:
SCHEDULE_NAME = "cloudtrail-risk-data-quality"
try:
    sm_client.delete_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)   # re-runs: replace the old schedule
    time.sleep(30)
except sm_client.exceptions.ResourceNotFound:
    pass

monitor.create_monitoring_schedule(
    monitor_schedule_name=SCHEDULE_NAME, endpoint_input=ENDPOINT_NAME,
    output_s3_uri=ctx.s3_uri("monitoring", "reports"),
    statistics=monitor.baseline_statistics(), constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.now(),          # production: CronExpressionGenerator.hourly()
    data_analysis_start_time="-PT1H", data_analysis_end_time="-PT0H",
    enable_cloudwatch_metrics=True,
)
aws.save_state(ctx, monitoring_schedule=SCHEDULE_NAME)

while True:
    executions = monitor.list_executions()
    status = executions[-1].describe()["ProcessingJobStatus"] if executions else "Pending"
    print(time.strftime("%H:%M:%S"), "monitoring execution:", status)
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(60)

In [ ]:
summary = sm_client.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME).get("LastMonitoringExecutionSummary", {})
print("Execution status:", summary.get("MonitoringExecutionStatus"), "|", summary.get("FailureReason", ""))
violations = monitor.latest_monitoring_constraint_violations()
if violations is not None:
    report = pd.DataFrame(violations.body_dict.get("violations", []))
    print(f"{len(report)} constraint violations -> {violations.file_s3_uri}")
    display(report[["feature_name", "constraint_check_type", "description"]].head(20) if len(report) else report)
else:
    print("No violation report yet - re-run this cell in a few minutes.")

**Reading the report.** The drifted requests break constraints learned from the training data (e.g. values far outside the observed range, and the never-seen region lands in the `last_region=__other__` column). In production these violations raise a CloudWatch metric that pages the on-call engineer and triggers an investigation of the upstream change and, if the change is genuine, **retraining through the CI/CD pipeline** (notebook 06).

## 4. Model and data monitoring report (offline, with delayed labels)
Model Monitor checks inputs. To monitor the **model** we also need its outputs and, once analysts have reviewed alerts, the labels. Here we replay the 2020 test year month by month as if it were production traffic and track:
* **feature drift** — Population Stability Index (PSI) of every input vs. the training period (< 0.10 stable, 0.10–0.25 warn, > 0.25 alert);
* **prediction drift** — PSI of the risk-score distribution vs. the validation year, and the daily alert volume vs. the analyst budget;
* **performance with delayed labels** — monthly recall and average precision.

In [ ]:
local_dir = DATA_DIR / "model"
booster = model.load_booster(local_dir / "xgboost-model")
train_encoded = encoder.transform(splits["train"])
validation_scores = model.predict(booster, encoder.transform(splits["validation"]))
test = splits["test"].copy()
test["score"] = model.predict(booster, encoder.transform(test))
test["month"] = test.window_start.dt.strftime("%Y-%m")

rows, psi_matrix = [], {}
for month, frame in test.groupby("month"):
    drift = evaluation.drift_report(train_encoded, encoder.transform(frame), config.NUMERIC_FEATURES)
    psi_matrix[month] = drift.set_index("feature")["psi"]
    y = frame[config.TARGET_COLUMN].to_numpy()
    alerts = frame.score >= thresholds["medium"]
    rows.append({"month": month, "windows": len(frame), "features_warn": int((drift.status == "WARN").sum()),
                 "features_alert": int((drift.status == "ALERT").sum()),
                 "score_psi": round(evaluation.psi(validation_scores, frame.score.to_numpy()), 3),
                 "alerts_per_day": round(alerts.sum() / evaluation.days_covered(frame.window_start), 2),
                 "suspicious": int(y.sum()),
                 "recall": round(alerts[y == 1].mean(), 3) if y.sum() else np.nan,
                 "average_precision": round(average_precision_score(y, frame.score), 3) if 0 < y.sum() < len(y) else np.nan})
monthly = pd.DataFrame(rows)
monthly

In [ ]:
psi_table = pd.DataFrame(psi_matrix)
top_features = psi_table.max(axis=1).sort_values(ascending=False).head(12).index
fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [1.3, 1]})
image = axes[0].imshow(psi_table.loc[top_features], cmap="Oranges", vmin=0, vmax=0.5, aspect="auto")
axes[0].set_xticks(range(psi_table.shape[1]), psi_table.columns, rotation=45)
axes[0].set_yticks(range(len(top_features)), top_features)
axes[0].grid(False)
axes[0].set_title("Feature drift (PSI vs. training) - 12 most drifting features")
fig.colorbar(image, ax=axes[0], shrink=0.8, label="PSI (0.25 = alert)")

axes[1].bar(monthly.month, monthly.alerts_per_day, color=BLUE, width=0.6, label="MEDIUM+ alerts per day")
axes[1].axhline(config.MEDIUM_ALERTS_PER_DAY, color="black", ls="--", lw=1, label="analyst budget")
axes[1].axhline(2 * config.MEDIUM_ALERTS_PER_DAY, color=ORANGE, ls=":", lw=1.5, label="alarm: 2x budget")
axes[1].set(title="Prediction volume vs. analyst capacity", ylabel="alerts per day")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="x", visible=False)
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.4))
known = monthly.dropna(subset=["recall"])
ax.plot(known.month, known.recall, "o-", color=BLUE, label="recall at MEDIUM threshold")
ax.plot(known.month, known.average_precision, "s-", color=ORANGE, label="average precision")
ax.set(title="Model performance by month once labels arrive (test year)", ylim=(0, 1.05), ylabel="metric")
for _, row in known.iterrows():
    ax.annotate(f"n={row.suspicious}", (row.month, row.recall), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7)
ax.legend(loc="lower left")
plt.show()

### 4.1 Monitoring plan (what fires, and what we do)

In [ ]:
pd.DataFrame([
    ("Infrastructure", "5xx errors / p95 latency / CPU", "> 0 errors in 5 min; p95 > 200 ms or CPU > 80 % for 15 min", "CloudWatch alarm -> on-call; scale out or roll back"),
    ("Data quality", "schema, completeness, ranges (Model Monitor)", "any constraint violation in an hourly run", "inspect upstream log change; block retraining on bad data"),
    ("Data drift", "PSI per input feature vs. training", "PSI > 0.25 on >= 5 features", "investigate; trigger retraining pipeline"),
    ("Prediction drift", "score PSI vs. validation; alerts per day", "score PSI > 0.25 or alerts > 2x analyst budget", "re-check thresholds; analyst review"),
    ("Model quality", "monthly recall / AP on analyst-reviewed alerts", "recall < 50 % or AP lift < 5x for 2 months", "retrain via CI/CD pipeline; manual approval before deploy"),
], columns=["layer", "metric", "threshold", "action"])

## 5. Cleanup — stop everything that bills by the hour
Run this after recording the video demo. It deletes the monitoring schedule, the endpoint, its configuration and model. The S3 data, the Model Registry versions, the CloudWatch dashboard and the feature group are kept as evidence for the submission (they cost nothing or cents). Set `DELETE_FEATURE_GROUP = True` at the very end of the project.

In [ ]:
CLEANUP = False               # <- set True when you are done with the demo
DELETE_FEATURE_GROUP = False  # <- set True at the very end of the project

if CLEANUP:
    if state.get("monitoring_schedule"):
        try:
            sm_client.delete_monitoring_schedule(MonitoringScheduleName=state["monitoring_schedule"])
            print("deleted monitoring schedule")
            time.sleep(30)
        except sm_client.exceptions.ResourceNotFound:
            pass
    try:
        config_name = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointConfigName"]
        model_names = [v["ModelName"] for v in sm_client.describe_endpoint_config(EndpointConfigName=config_name)["ProductionVariants"]]
        sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
        sm_client.delete_endpoint_config(EndpointConfigName=config_name)
        for name in model_names:
            sm_client.delete_model(ModelName=name)
        print("deleted endpoint, endpoint config and model")
    except sm_client.exceptions.ClientError as error:
        print("endpoint already deleted:", error)
    if DELETE_FEATURE_GROUP and state.get("feature_group"):
        sm_client.delete_feature_group(FeatureGroupName=state["feature_group"])
        print("deleted feature group")
    print("Remaining endpoints:", [e["EndpointName"] for e in sm_client.list_endpoints()["Endpoints"]])

## 6. Summary
* **Infrastructure:** CloudWatch dashboard (invocations, errors, latency, CPU/memory/disk) and three alarms.
* **Data:** Model Monitor baseline + data-quality execution on captured traffic; the synthetic drift produced constraint violations.
* **Model:** monthly feature-drift (PSI), prediction-drift and delayed-label performance report, with a documented threshold → action plan.